In [ ]:
import requests
import pandas as pd
from tqdm import tqdm

key = "1162d4704019475891b562285068b555"
base = "https://api.rawg.io/api"

def fetch_games(page, page_size=40):
    try:
        r = requests.get(f"{base}/games",
            params={"key": key, "page": page, "page_size": page_size, "ordering": "-added"})

        if r.status_code in [429, 403]:
            return "LIMIT"
        return r.json()
    except:
        return "ERROR"


def fetch_count(url):
    try:
        r = requests.get(url)
        if r.status_code in [429, 403]:
            return "LIMIT"
        data = r.json()
        return len(data.get("results", []))
    except:
        return 0


def build_dataset(target_size=1000):
    all_data = []
    page = 1
    last_page_completed = 0
    pbar = tqdm(total=target_size)

    try:
        while len(all_data) < target_size:
            data = fetch_games(page)

            if data in ["LIMIT", "ERROR"]:
                print("\nStopping:", data)
                break

            games = data.get("results", [])
            for game in games:
                if len(all_data) >= target_size:
                    break

                game_id = game["id"]

                screenshots_url = f"{base}/games/{game_id}/screenshots?key={key}"
                screenshots_count = fetch_count(screenshots_url)

                genres = ", ".join([g["name"] for g in game.get("genres", [])])
                platforms = ", ".join([p["platform"]["name"] for p in game.get("platforms", [])])

                all_data.append({
                    "id": game_id,
                    "name": game.get("name"),
                    "released": game.get("released"),
                    "rating": game.get("rating"),
                    "ratings_count": game.get("ratings_count"),
                    "metacritic": game.get("metacritic"),
                    "playtime": game.get("playtime"),
                    "genres": genres,
                    "platforms": platforms,
                    "screenshots_count": screenshots_count})

                pbar.update(1)

            last_page_completed = page
            page += 1

    except Exception as e:
        print("\nStopped early:", e)

    finally:
        pbar.close()

        df = pd.DataFrame(all_data)
        df.to_csv("rawg_games_part10.csv", index=False)

        print("\nSaved rows:", len(df))
        print("Last completed page:", last_page_completed)
        return df

df = build_dataset(target_size=3000)
print(df.shape)

  1%|▌                                        | 40/3000 [00:13<18:02,  2.73it/s]